# 03 · Joins (BigQuery)

A **join** puts columns from two tables side by side, by matching a key.

- `INNER JOIN`: only rows that match on both sides
- `LEFT JOIN`: all rows from the left table, NULLs where the right side has no match
- `FULL OUTER JOIN`: all rows from both sides
- **anti join**: rows with **no** match (LEFT JOIN + `IS NULL`, or `NOT EXISTS`)
- **semi join**: rows that have **at least one** match, without repeating them (`EXISTS` / `IN`)

**Fan-out** = a join makes more rows than you expect (one order × many items). Then SUM gives a too-big number.

**How to run:** Start each SQL cell with `%%bigquery`. Write tables as `shield.table_name`.
Each cell prints how much data it read. `%%bigquery dry` checks the size for free.

Tables: see [tables_guide.md](../../jarvis/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`bigquery/practice/03_joins.ipynb`](../../bigquery/practice/03_joins.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "bigquery"))
%load_ext bq_helper

## Q1. Employee with department name  ·  🟢 easy

Show `name` and `dept_name` for every employee who **has** a department.

*Expected result: 29 rows · columns: `name`, `dept_name`*

**How it works:** Inner join on `dept_id`: each employee gets the matching department row. The employee with NULL `dept_id` has no match and drops out.

In [ ]:
%%bigquery
SELECT e.name, d.dept_name
FROM shield.employees e
JOIN shield.departments d ON d.dept_id = e.dept_id

,name,dept_name
0,Alice Johnson,Engineering
1,Bob Smith,Engineering
2,Carol White,Engineering
3,David Brown,Engineering
4,Eva Green,Engineering
5,Frank Moore,Engineering
6,Grace Lee,Engineering
7,Henry Clark,Sales
8,Irene Lewis,Sales
9,Jack Walker,Sales


## Q2. Keep employees with no department  ·  🟢 easy

Show `name` and `dept_name` for **all** employees. The one with no department must show NULL.

*Expected result: 30 rows · columns: `name`, `dept_name`*

**How it works:** LEFT JOIN keeps every employee. When there is no matching department, the department columns are NULL.

In [ ]:
%%bigquery
SELECT e.name, d.dept_name
FROM shield.employees e
LEFT JOIN shield.departments d ON d.dept_id = e.dept_id

,name,dept_name
0,Alice Johnson,Engineering
1,Bob Smith,Engineering
2,Carol White,Engineering
3,David Brown,Engineering
4,Eva Green,Engineering
5,Frank Moore,Engineering
6,Grace Lee,Engineering
7,Henry Clark,Sales
8,Irene Lewis,Sales
9,Jack Walker,Sales


## Q3. Department with no employees  ·  🟢 easy

Show the `dept_name` of departments that have **no** employees.

*Expected result: 1 row · columns: `dept_name`*

**How it works:** LEFT JOIN departments → employees, then keep rows where the employee side is NULL = departments with no match (anti join).

In [ ]:
%%bigquery
SELECT d.dept_name
FROM shield.departments d
LEFT JOIN shield.employees e ON e.dept_id = d.dept_id
WHERE e.emp_id IS NULL

,dept_name
0,Legal


**Why / traps:** This is an **anti join**. Test the right table's key (`e.emp_id IS NULL`), not a column that can be NULL on its own.

## Q4. Customers who never ordered  ·  🟢 easy

Show `customer_id` and `name` of customers with **no** orders. Use `NOT EXISTS`.

*Expected result: 8 rows · columns: `customer_id`, `name`*

**How it works:** `NOT EXISTS` checks, for each customer, that no order row has their id. Customers 33–40 pass.

In [ ]:
%%bigquery
SELECT c.customer_id, c.name
FROM shield.customers c
WHERE NOT EXISTS (SELECT 1 FROM shield.orders o WHERE o.customer_id = c.customer_id)
ORDER BY c.customer_id

,customer_id,name
0,33,Lucas Lopez
1,34,Mei Kim
2,35,Omar Khan
3,36,Hannah Muller
4,37,Ravi Garcia
5,38,Elena Rossi
6,39,Kenji Chen
7,40,Grace Patel


**Why / traps:** `NOT EXISTS` is safe with NULLs. `NOT IN (subquery)` returns **nothing** if the subquery has even one NULL (see topic 04).

## Q5. Employee and manager names (self join)  ·  🟢 easy

Show `employee` (name) and `manager` (manager's name) for **all** employees. Top bosses have no manager: show NULL.

*Expected result: 30 rows · columns: `employee`, `manager`*

**How it works:** Join the table to itself: `e` = the employee, `m` = the manager, matched by `m.emp_id = e.manager_id`. LEFT JOIN keeps the top bosses.

In [ ]:
%%bigquery
SELECT e.name AS employee, m.name AS manager
FROM shield.employees e
LEFT JOIN shield.employees m ON m.emp_id = e.manager_id

,employee,manager
0,Alice Johnson,None
1,Bob Smith,Alice Johnson
2,Carol White,Bob Smith
3,David Brown,Bob Smith
4,Eva Green,Bob Smith
5,Frank Moore,Carol White
6,Grace Lee,Carol White
7,Henry Clark,Alice Johnson
8,Irene Lewis,Henry Clark
9,Jack Walker,Henry Clark


**Why / traps:** A **self join** joins a table to itself. Give each copy its own alias.

## Q6. Close to the manager's pay  ·  🟡 medium

Show `employee`, `salary`, `manager`, `manager_salary` for employees who earn **at least 90%** of their manager's salary.

(Try first: who earns **more** than their manager? In this data: nobody, 0 rows.)

*Expected result: 1 row · columns: `employee`, `salary`, `manager`, `manager_salary`*

**How it works:** Self join employee → manager, then compare the two salaries in WHERE (`e.salary >= 0.9 * m.salary`).

In [ ]:
%%bigquery
SELECT e.name AS employee, e.salary,
       m.name AS manager, m.salary AS manager_salary
FROM shield.employees e
JOIN shield.employees m ON m.emp_id = e.manager_id
WHERE e.salary >= 0.9 * m.salary

,employee,salary,manager,manager_salary
0,Uma Perez,98000.000000000,Sam Carter,100000.000000000


## Q7. Everyone and every department (FULL OUTER JOIN)  ·  🟡 medium

Show `dept_name` and `name` (employee). Keep departments with no employees **and** employees with no department.

*Expected result: 31 rows · columns: `dept_name`, `name`*

**How it works:** FULL OUTER JOIN keeps matches + unmatched rows from **both** sides: Legal (no employees) and Daniel (no department).

In [ ]:
%%bigquery
SELECT d.dept_name, e.name
FROM shield.departments d
FULL OUTER JOIN shield.employees e ON e.dept_id = d.dept_id

,dept_name,name
0,Engineering,Alice Johnson
1,Engineering,Bob Smith
2,Engineering,Carol White
3,Engineering,David Brown
4,Engineering,Eva Green
5,Engineering,Frank Moore
6,Engineering,Grace Lee
7,Sales,Henry Clark
8,Sales,Irene Lewis
9,Sales,Jack Walker


**Why / traps:** 29 matches + 1 department alone (Legal) + 1 employee alone (Daniel) = 31 rows.

## Q8. Products never sold  ·  🟢 easy

Show `product_id` and `product_name` of products that are not in `order_items`.

*Expected result: 2 rows · columns: `product_id`, `product_name`*

**How it works:** LEFT JOIN products → order_items and keep rows with no item = never sold.

In [ ]:
%%bigquery
SELECT p.product_id, p.product_name
FROM shield.products p
LEFT JOIN shield.order_items i ON i.product_id = p.product_id
WHERE i.product_id IS NULL

,product_id,product_name
0,19,Yoga Mat
1,20,Dumbbell Set


## Q9. Customers who bought a book (semi join)  ·  🟡 medium

Show `customer_id` and `name` of customers who bought **at least one** product in category `Books` (any order status). Each customer **once**.

*Expected result: 23 rows · columns: `customer_id`, `name`*

**How it works:** For each customer, `EXISTS` looks for at least one order line with a Books product (orders → order_items → products). Each customer appears once.

In [ ]:
%%bigquery
SELECT c.customer_id, c.name
FROM shield.customers c
WHERE EXISTS (
    SELECT 1
    FROM shield.orders o
    JOIN shield.order_items i ON i.order_id = o.order_id
    JOIN shield.products p    ON p.product_id = i.product_id
    WHERE o.customer_id = c.customer_id
      AND p.category = 'Books')

,customer_id,name
0,15,Omar Khan
1,19,Kenji Chen
2,20,Grace Patel
3,26,Fatima Muller
4,1,Aarav Nguyen
5,18,Elena Rossi
6,23,John Lopez
7,24,Maria Kim
8,30,Sofia Patel
9,31,Noah Nguyen


**Why / traps:** If you JOIN instead of EXISTS, a customer with 3 book rows shows 3 times and you need DISTINCT. EXISTS (a **semi join**) never repeats rows.

## Q10. Revenue per category (fan-out from duplicates)  ·  🔴 hard

Show `category` and `revenue` = SUM(quantity × price) for **completed** orders. Careful: `orders` has duplicate rows. If you join to them, items get counted twice.

*Expected result: 4 rows · columns: `category`, `revenue`*

**How it works:** 1) Make orders unique per order_id first (DISTINCT). 2) Join to order_items and products. 3) Keep completed. 4) SUM(quantity × price) per category.

In [ ]:
%%bigquery
SELECT p.category, SUM(i.quantity * p.price) AS revenue
FROM (SELECT DISTINCT order_id, status FROM shield.orders) o
JOIN shield.order_items i ON i.order_id = o.order_id
JOIN shield.products p    ON p.product_id = i.product_id
WHERE o.status = 'completed'
GROUP BY p.category
ORDER BY revenue DESC

,category,revenue
0,Electronics,2590.000000000
1,Books,2005.000000000
2,Clothing,1055.000000000
3,Home,707.000000000


**Why / traps:** Before a join, ask: *is the key unique on this side?* Here `order_id` is **not** unique in `orders`, so make it unique first (DISTINCT or GROUP BY).

## Q11. Amount and items per customer (fan-out)  ·  🔴 hard

For each customer with orders, show `customer_id`, `total_amount` (sum of order amounts, each order once) and `total_items` (sum of quantity). Joining orders to items first and then summing amount gives a wrong, too-big number. Avoid it.

*Expected result: 32 rows · columns: `customer_id`, `total_amount`, `total_items`*

**How it works:** Aggregate **each side to the order level first** (amount per order, quantity per order), then join 1-to-1 and sum per customer. No row is counted twice.

In [ ]:
%%bigquery
WITH o AS (
    SELECT customer_id, order_id, SUM(amount) AS amount
    FROM (SELECT DISTINCT order_id, customer_id, amount FROM shield.orders) d
    GROUP BY customer_id, order_id
),
items AS (
    SELECT order_id, SUM(quantity) AS qty
    FROM shield.order_items
    GROUP BY order_id
)
SELECT o.customer_id,
       SUM(o.amount) AS total_amount,
       SUM(items.qty) AS total_items
FROM o
LEFT JOIN items ON items.order_id = o.order_id
GROUP BY o.customer_id

,customer_id,total_amount,total_items
0,2,135.000000000,5
1,11,65.000000000,5
2,18,309.000000000,10
3,22,364.000000000,11
4,23,315.000000000,10
5,24,248.000000000,15
6,26,904.000000000,22
7,27,892.000000000,31
8,30,1265.000000000,35
9,31,575.000000000,17


**Why / traps:** Fix for fan-out: **aggregate each side to the join key first**, then join 1-to-1.

## Q12. Every category × every year (CROSS JOIN)  ·  🔴 hard

Show `category`, `order_year` (2023, 2024, 2025) and `items_sold` = SUM(quantity) for completed orders. Show **every** category-year pair, with **0** when nothing was sold.

*Expected result: 15 rows · columns: `category`, `order_year`, `items_sold`*

**How it works:** 1) Build all categories and all years. 2) CROSS JOIN = every pair. 3) LEFT JOIN the real sales per pair. 4) COALESCE missing sales to 0.

In [ ]:
%%bigquery
WITH years AS (
    SELECT DISTINCT EXTRACT(YEAR FROM order_date) AS order_year FROM shield.orders
),
cats AS (
    SELECT DISTINCT category FROM shield.products
),
sold AS (
    SELECT p.category, EXTRACT(YEAR FROM o.order_date) AS order_year, SUM(i.quantity) AS qty
    FROM (SELECT DISTINCT order_id, order_date, status FROM shield.orders) o
    JOIN shield.order_items i ON i.order_id = o.order_id
    JOIN shield.products p    ON p.product_id = i.product_id
    WHERE o.status = 'completed'
    GROUP BY p.category, EXTRACT(YEAR FROM o.order_date)
)
SELECT c.category, y.order_year, COALESCE(s.qty, 0) AS items_sold
FROM cats c
CROSS JOIN years y
LEFT JOIN sold s ON s.category = c.category AND s.order_year = y.order_year
ORDER BY c.category, y.order_year

,category,order_year,items_sold
0,Books,2023,7
1,Books,2024,25
2,Books,2025,23
3,Clothing,2023,5
4,Clothing,2024,14
5,Clothing,2025,26
6,Electronics,2023,2
7,Electronics,2024,23
8,Electronics,2025,27
9,Home,2023,5


**Why / traps:** CROSS JOIN = every row × every row (5 categories × 3 years = 15 rows). Build the full grid first, then LEFT JOIN the facts. Same trick fills missing dates.

## Q13. Same department, same salary (pairs)  ·  🟡 medium

Find pairs of employees in the **same department** with the **same salary**. Show `dept_id`, `emp1`, `emp2`, `salary`. Each pair once (not A-B and B-A).

*Expected result: 8 rows · columns: `dept_id`, `emp1`, `emp2`, `salary`*

**How it works:** Self join on same department **and** same salary; `b.emp_id > a.emp_id` keeps each pair once and removes pairing a person with themselves.

In [ ]:
%%bigquery
SELECT a.dept_id, a.name AS emp1, b.name AS emp2, a.salary
FROM shield.employees a
JOIN shield.employees b
  ON b.dept_id = a.dept_id
 AND b.salary  = a.salary
 AND b.emp_id  > a.emp_id
ORDER BY a.dept_id, emp1, emp2

,dept_id,emp1,emp2,salary
0,1,Carol White,David Brown,120000.000000000
1,1,Frank Moore,Grace Lee,95000.000000000
2,2,Irene Lewis,Jack Walker,90000.000000000
3,2,Irene Lewis,Karen Hall,90000.000000000
4,2,Jack Walker,Karen Hall,90000.000000000
5,3,Olivia Scott,Paul Adams,85000.000000000
6,4,Tina Mitchell,Uma Perez,98000.000000000
7,6,Zane Parker,Amy Evans,75000.000000000


**Why / traps:** `b.emp_id > a.emp_id` removes self-pairs (A-A) and mirror pairs (B-A). A 3-way tie (Sales, 90,000) makes 3 pairs.